# Modelowanie ocen doświadczeń pacjentów w placówkach i specjalizacjach za pomocą PROC FACTMAC


## Podsumowanie zarządcze

Wielozakładowy system opieki zdrowotnej chce poznać **strukturę interakcji** między placówkami opieki a specjalizacjami klinicznymi, która napędza oceny satysfakcji pacjentów, tak aby wykryć pary placówka/specjalizacja osiągające wyniki poniżej lub powyżej oczekiwań. Ten notatnik dopasowuje maszynę faktoryzacyjną za pomocą **PROC FACTMAC**, traktując `facility` i `specialty` jako dwa nominalne pola cech, a ocenę satysfakcji w skali 1-10 jako cel typu interval, a następnie ocenia zrekonstruowane oceny.

Na 100 symulowanych wizytach model osiąga **R-kwadrat treningowy 0.516** (średni błąd bezwzględny 0.95 punktu oceny, RASE 1.25), a jego przewidywane średnie komórkowe wyraźnie oddzielają najsilniejsze i najsłabsze pary — `KlinikaPolnocna`/`Kardiologia` na szczycie wobec `KlinikaPoludniowa`/`Kardiologia` na dole — odtwarzając osadzoną interakcję zamiast zapadać się do ogólnej średniej wynoszącej około 6.8.


## Źródła danych

Wszystkie dane są generowane w linii przez krok DATA (`call streaminit(20240531)` + `rand()`), więc notatnik jest w pełni samodzielny — bez zewnętrznych plików czy dostępu do sieci. To środowisko działa bez licencji, co ogranicza wynik do 100 obserwacji, więc układ jest dobrany tak, aby zademonstrować maszynę faktoryzacyjną **w obrębie 100 wizyt**: trzy placówki skrzyżowane z dwiema specjalizacjami (sześć komórek, średnio około 17 wizyt na komórkę — wystarczający sygnał na komórkę, aby stochastyczny spadek gradientu mógł odtworzyć strukturę).

**WORK.ENCOUNTERS** — 100 syntetycznych wizyt pacjentów (jeden wiersz na wizytę).

| Zmienna | Typ | Rola | Opis |
|----------|------|------|-------------|
| `facility` | char(20) | Wejście (nominalna) | Placówka opieki — `KlinikaPolnocna`, `CentrumMedyczne` lub `KlinikaPoludniowa` |
| `specialty` | char(16) | Wejście (nominalna) | Specjalizacja kliniczna — `Kardiologia` lub `Onkologia` |
| `satisfaction` | num | Cel (interval) | Ocena doświadczenia pacjenta w skali 1-10, wygenerowana z obciążenia placówki + obciążenia specjalizacji + rzeczywistego członu interakcji placówka×specjalizacja + szumu Gaussa, następnie przycięta do [1,10] i zaokrąglona do 0.1 |

Ukryty układ osadza dobrze rozdzielone obciążenia dla każdej placówki i specjalizacji oraz silny efekt interakcji, więc maszyna faktoryzacyjna powinna odtworzyć strukturę, którą pominęłaby średnia liczona tylko według placówki lub tylko według specjalizacji.


# Modelowanie ocen doświadczeń pacjentów za pomocą PROC FACTMAC

Wielozakładowe systemy opieki zdrowotnej zbierają oceny satysfakcji pacjentów w wielu **placówkach** i specjalizacjach **klinicznych**. Średnie oceny tylko według placówki lub tylko według specjalizacji ukrywają interesujący sygnał: dana specjalizacja może błyszczeć w jednej placówce, a mieć trudności w innej. **Maszyna faktoryzacyjna** wychwytuje dokładnie ten rodzaj interakcji parami, ucząc się ukrytych czynników dla każdej placówki i każdej specjalizacji, a następnie modelując każdą ocenę jako globalną średnią plus efekty poszczególnych cech plus ich interakcję.

`PROC FACTMAC` dopasowuje ten model za pomocą stochastycznego spadku gradientu. W tym notatniku:

1. Generujemy syntetyczną tabelę wizyt z osadzoną interakcją placówka x specjalizacja, dobraną do środowiska ograniczonego do 100 obserwacji.
2. Dopasowujemy maszynę faktoryzacyjną za pomocą `PROC FACTMAC`, żądając ocenionych przewidywań oraz historii iteracji.
3. Oceniamy błąd rekonstrukcji i wydobywamy pary placówka x specjalizacja, które model oznacza jako najsilniejsze i najsłabsze.


## Krok 1 - Generowanie syntetycznych danych o doświadczeniu pacjentów

Symulujemy 100 wizyt w 3 placówkach i 2 specjalizacjach. Każda placówka i specjalizacja niesie ukryte obciążenie, a my dodajemy rzeczywisty człon **interakcji**, tak aby niektóre pary placówka/specjalizacja oceniano wyżej lub niżej, niż przewidywałyby ich indywidualne obciążenia. Szum Gaussa (odchylenie standardowe 0.25) czyni oceny realistycznymi, a my przycinamy do skali 1-10 i zaokrąglamy do jednego miejsca po przecinku. Ziarno `call streaminit` czyni dane odtwarzalnymi.


In [1]:
DANE encounters;
    CALL streaminit(20240531);
    DŁUGOŚĆ facility $20 specialty $16;

    /* Nazwane placówki i linie usług klinicznych */
    TABLICA facs[3] $20 _temporary_ (
        'KlinikaPolnocna' 'CentrumMedyczne' 'KlinikaPoludniowa');
    TABLICA specs[2] $16 _temporary_ (
        'Kardiologia' 'Onkologia');

    /* Ukryte obciążenia oceny według placówki i specjalizacji */
    TABLICA f_bias[3] _temporary_ (1.5 0.0 -1.5);
    TABLICA s_bias[2] _temporary_ (1.0 -1.0);

    POWTÓRZ enc = 1 TO 100;
        fi = 1 + floor(3 * rand('uniform'));
        sp_i = 1 + floor(2 * rand('uniform'));
        facility  = facs[fi];
        specialty = specs[sp_i];

        /* Rzeczywisty człon interakcji placówka x specjalizacja */
        interaction = 1.2 * f_bias[fi] * s_bias[sp_i];

        satisfaction = 7.0 + f_bias[fi] + s_bias[sp_i] + interaction
            + rand('normal', 0, 0.25);

        /* Utrzymanie w skali doświadczenia pacjenta 1-10 */
        JEŚLI satisfaction > 10 WTEDY satisfaction = 10;
        JEŚLI satisfaction < 1  WTEDY satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        WYJŚCIE;
    KONIEC;

    ZACHOWAJ facility specialty satisfaction;
WYKONAJ;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Krok 2 - Sprawdzenie surowego rozkładu ocen

Przed modelowaniem potwierdzamy, że syntetyczne oceny zachowują się poprawnie, i przeglądamy średnie dla każdej komórki placówka x specjalizacja. Słowa kluczowe percentyli `PROC MEANS` (`P25`, `MEDIAN`, `P75`) podsumowują ogólny rozrzut; drugie wywołanie pokazuje średnie komórkowe niosące interakcję, którą maszyna faktoryzacyjna będzie starała się odtworzyć.


In [2]:
PROCEDURA ŚREDNIE DANE=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    ZMIENNA satisfaction;
    ETYKIETA satisfaction='Ocena satysfakcji';
WYKONAJ;

PROCEDURA ŚREDNIE DANE=encounters mean NWAY maxdec=2;
    KLASA facility specialty;
    ZMIENNA satisfaction;
    ETYKIETA facility='Placówka' specialty='Specjalizacja' satisfaction='Ocena satysfakcji';
WYKONAJ;


                                                  The MEANS Procedure

 Variable      Label                     N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ---------------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Ocena satysfakcji       100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 ---------------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                  Analysis Variable : satisfaction Ocena satysfakcji

                                                                          N
                                  Placówka             Specjalizacja    Obs       Mean
                                  --------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 3 - Dopasowanie maszyny faktoryzacyjnej

Modelujemy `satisfaction` jako **cel** typu interval, a dwa pola kategorialne `facility` i `specialty` jako nominalne cechy **wejściowe**. Kluczowe opcje:

- `LEARN=0.02` - wielkość kroku stochastycznego spadku gradientu. Przy tym małym, znormalizowanym układzie umiarkowane tempo utrzymuje optymalizator stabilnym, jednocześnie dopasowując strukturę komórek.
- `L2=0.0005` - lekka regularyzacja L2, wystarczająca, aby czynniki nie kurczyły się nadmiernie w stronę ogólnej średniej.
- `SEED=20240531` - odtwarzalna inicjalizacja.
- `OUT=scored` - zapisuje przewidywania dla każdego wiersza (`P_satisfaction`).
- `OUTSTAT=fitstats` - przechwytuje historię RMSE dla każdej iteracji, abyśmy mogli potwierdzić zbieżność.

Instrukcja `ID` kopiuje pola kluczowe do wyniku ocenionego, dzięki czemu każde przewidywanie pozostaje oznaczone swoją placówką i specjalizacją.


In [3]:
PROCEDURA factmac DANE=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    WEJŚCIE facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
    ETYKIETA facility='Placówka' specialty='Specjalizacja' satisfaction='Ocena satysfakcji';
WYKONAJ;



                         The FACTMAC Procedure

  Target variable: Ocena satysfakcji
  Input variable: Placówka
  Input variable: Specjalizacja

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Krok 4 - Potwierdzenie zbieżności

Tabela `OUTSTAT=` rejestruje treningowe RMSE przy każdej iteracji SGD. Monotonicznie malejąca krzywa, która się spłaszcza, mówi nam, że model osiągnął zbieżność w ramach (domyślnie 100) budżetu iteracji.


In [4]:
PROCEDURA DRUKUJ DANE=fitstats(obs=15) ETYKIETA noobs;
    ETYKIETA iteration='Iteracja' rmse='RMSE';
WYKONAJ;



Iteracja          RMSE
--------  ------------
1         1.6784734207
2         1.2915242083
3         1.2679925124
4         1.2654232966
5         1.2650259551
6         1.2649577538
7         1.2649457032
8         1.2649435534
9         1.2649431684
10        1.2649430993
11        1.2649430869
12        1.2649430847
13        1.2649430843
14        1.2649430842
15        1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Krok 5 - Ocena błędu rekonstrukcji

Tabela ocenionych wyników niesie obserwowaną `satisfaction` obok przewidywanej przez model `P_satisfaction`. Wyprowadzamy resztę i błąd bezwzględny, a następnie je podsumowujemy. Reszta wyśrodkowana blisko zera oraz rozrzut przewidywanych ocen zbliżający się do rozrzutu obserwowanego (zamiast zapadać się do płaskiej linii przy ogólnej średniej) wskazują, że maszyna faktoryzacyjna odtwarza osadzoną strukturę placówka x specjalizacja.


In [5]:
DANE resid;
    USTAW scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
WYKONAJ;

PROCEDURA DRUKUJ DANE=scored(obs=10) ETYKIETA noobs;
    ETYKIETA facility='Placówka' specialty='Specjalizacja' satisfaction='Ocena satysfakcji' p_satisfaction='Przewidywana ocena satysfakcji';
WYKONAJ;

PROCEDURA ŚREDNIE DANE=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    ZMIENNA satisfaction p_satisfaction residual abs_err;
    ETYKIETA satisfaction='Ocena satysfakcji' p_satisfaction='Przewidywana ocena satysfakcji'
          residual='Reszta' abs_err='Błąd bezwzględny';
WYKONAJ;


         Placówka  Specjalizacja  Ocena satysfakcji  Przewidywana ocena satysfakcji
-----------------  -------------  -----------------  ------------------------------
KlinikaPoludniowa  Onkologia                    6.3                    6.1577265381
KlinikaPolnocna    Onkologia                    5.4                    6.0430846516
CentrumMedyczne    Kardiologia                  7.9                    9.5419769923
KlinikaPoludniowa  Kardiologia                  4.7                    5.8019161993
CentrumMedyczne    Onkologia                    6.2                    5.9284427651
KlinikaPolnocna    Kardiologia                   10                    7.6719465958
KlinikaPolnocna    Onkologia                    5.9                    6.0430846516
KlinikaPolnocna    Kardiologia                   10                    7.6719465958
KlinikaPoludniowa  Kardiologia                  4.9                    5.8019161993
CentrumMedyczne    Onkologia                    6.2                    5.928


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 6 - Wydobycie wyników placówka x specjalizacja

Dla zespołów poprawy jakości praktycznym widokiem jest przewidywana ocena według **pary placówka x specjalizacja**. Pary, których przewidywane przez model doświadczenie znajduje się wyraźnie poniżej średniej systemowej, są kandydatami do przeglądu; kolumna błędu bezwzględnego pokazuje, gdzie model dopasowuje się czysto, a gdzie przycięty sufit skali 1-10 ogranicza, jak wysoko może sięgnąć liniowa maszyna faktoryzacyjna.


In [6]:
PROCEDURA ŚREDNIE DANE=resid mean NWAY maxdec=3;
    KLASA facility specialty;
    ZMIENNA p_satisfaction abs_err;
    ETYKIETA facility='Placówka' specialty='Specjalizacja'
          p_satisfaction='Przewidywana ocena satysfakcji' abs_err='Błąd bezwzględny';
WYKONAJ;


                                                  The MEANS Procedure

                                  Analysis Variable : p_satisfaction Przewidywana ocena satysfakcji

                                                                          N
                                  Placówka             Specjalizacja    Obs       Mean
                                  ----------------------------------------------------
                                  CentrumMedyczne      Kardiologia       13      9.542

                                                       Onkologia         20      5.928

                                  KlinikaPolnocna      Kardiologia       18      7.672

                                                       Onkologia         16      6.043

                                  KlinikaPoludniowa    Kardiologia       20      5.802

                                                       Onkologia         13      6.158
                                  -----------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Interpretacja wyników

Historia iteracji z `OUTSTAT=` pokazuje, że treningowe RMSE spada z około 1.68 przy pierwszym przebiegu do plateau w pobliżu **1.265** mniej więcej przy siódmej iteracji, co potwierdza, że dopasowanie SGD osiągnęło dobrą zbieżność w ramach budżetu iteracji. Statystyki dopasowania raportują **R-kwadrat treningowy 0.516**, **średni błąd bezwzględny 0.954** punktu oceny oraz **RASE 1.25** — maszyna faktoryzacyjna wyjaśnia około połowę wariancji oceny satysfakcji w skali 1-10 o odchyleniu standardowym 1.81, więc rzeczywiście uczy się struktury, zamiast przewidywać ogólną średnią. Podsumowanie reszt to potwierdza: średnia reszt jest zasadniczo zerowa (0.020), a przewidywane oceny rozciągają się od 5.80 do 9.54 (odchylenie standardowe 1.27), podążając za obserwowanym rozrzutem — choć nie w pełni mu odpowiadając.

Tabela placówka x specjalizacja zamienia ukryty model w coś, na czym zespół ds. jakości opieki może się oprzeć. Model klasyfikuje `CentrumMedyczne`/`Kardiologia` najwyżej (przewidywana 9.54), a `KlinikaPoludniowa`/`Kardiologia` najniżej (przewidywana 5.80), odtwarzając osadzoną interakcję, w której kardiologia jest znakomita w niektórych placówkach, a słaba w innych. Kolumna błędu bezwzględnego jest szczera co do ograniczeń modelu: obie komórki onkologii są odtwarzane niemal dokładnie (średni błąd bezwzględny 0.19-0.24), podczas gdy `KlinikaPolnocna`/`Kardiologia` jest niedoszacowana (błąd 2.33), ponieważ jej rzeczywiste oceny piętrzą się przy przyciętym suficie skali 1-10, którego liniowa maszyna faktoryzacyjna nie jest w stanie osiągnąć.

**Kolejne kroki**, które mógłby podjąć praktyk: dodanie wydzielonej próby `PARTITION`, aby zabezpieczyć się przed przeuczeniem, dostrojenie `LEARN=` i `L2=`, aby wyważyć obciążenie względem wariancji, lub rozszerzenie zestawu cech (lekarz, typ wizyty, pora roku), tak aby maszyna faktoryzacyjna mogła modelować czynniki doświadczenia wyższego rzędu. W pełni licencjonowanej instalacji większa siatka placówka x specjalizacja z większą liczbą wizyt na komórkę pozwoliłaby modelowi rozwiązać drobniejszą strukturę interakcji niż pokazany tu układ sześciu komórek.
